In [ ]:
import json
import pandas as pd
import numpy as np
import os
import csv
from sklearn.model_selection import train_test_split
import nltk

import os
os.getcwd()

huggingface_cache_dir = 'model'
os.getcwd()

os.getcwd()
from ftfy import fix_text
from ftfy import fix_encoding


In [ ]:
df = pd.read_csv('analyses/NOS/actors_NER/actors_training_df_final_quoted.csv',sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)
df['article_id'] = df['article_id'].astype(int)
df['entity_name'] = df['entity_name'].str.strip()
df["entity_name"] = df["entity_name"].apply(fix_text)
df["entity_name"] = df["entity_name"].apply(fix_encoding)

df.head()


In [ ]:
# read the reliability df
reliability_df = pd.read_csv('reliability_actors_final_cleaned_researcher.csv',
                             sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)

reliability_df['article_id'] = reliability_df['article_id'].astype(int)

reliability_df.head()

In [ ]:
reliability_df = reliability_df[reliability_df['coder'] == 'researcher']
print(reliability_df.shape)

In [ ]:
# limit df only to article_ids that are not in the test_df
train_df = df[~df['article_id'].isin(reliability_df['article_id'])]
print(train_df.shape)

In [ ]:
train_df.quoted.value_counts()

In [ ]:
train_df.actor_function.value_counts(dropna=False)

# Text preprocessing

In [ ]:
train_df.isnull().sum()

In [ ]:
print(train_df['input_text'].values[1])

# create a text preprocessing function where you lowercase the text and then lemmitize the text
def text_lower(text):
    text = text.lower()
    return text

train_df['input_text_lower'] = train_df['input_text'].apply(text_lower)
print(train_df['input_text_lower'].values[1])

In [ ]:
train_df.head()

# Tf-IDF + SVM

In [ ]:
# Select your features and target variable
X = train_df[['input_text_lower']]  # This should remain a DataFrame
y = train_df['actor_function']

# Split the data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, shuffle=True, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")  # Should be (n_samples_train, n_features)
print(f"y_train shape: {y_train.shape}")  # Should be (n_samples_train,)


In [ ]:
# see the distribution of the target variable
y_train.value_counts()

In [ ]:
# shape of the validation set
print(f"X_val shape: {X_val.shape}")  # Should be (n_samples_val, n_features)
print(f"y_val shape: {y_val.shape}")  # Should be (n_samples_val,)

In [ ]:
y_val.value_counts()

In [ ]:
nltk.download('stopwords')
stopwords = nltk.corpus.stopwords.words('dutch')

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB

# Define the parameter grid
param_grid_svc = {
    'tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)],
    'tfidf__max_features': [100, 2000, 1000, 5000, 10000],
    'clf__C': [0.1, 1, 10, 50, 100],
    'clf__max_iter': [50, 100, 500, 1000]
}

from sklearn.model_selection import StratifiedKFold


# Define the SVC pipeline
pipeline_svc = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 1), analyzer='word', stop_words=stopwords)),
    ('clf', SVC(decision_function_shape='ovo', random_state=42))
])

# Perform grid search for SVC
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search_svc = GridSearchCV(pipeline_svc, param_grid_svc, cv=stratified_cv, scoring='f1_macro', n_jobs=-1)
# grid_search_svc = GridSearchCV(pipeline_svc, param_grid_svc, cv=5, scoring='f1_macro', n_jobs=-1)
grid_search_svc.fit(X_train['input_text_lower'], y_train)  

print(f"Best parameters for SVC: {grid_search_svc.best_params_}")
print(f"Best score for SVC: {grid_search_svc.best_score_}")

In [ ]:
# Get the best parameters and the best model
best_params = grid_search_svc.best_params_
best_model = grid_search_svc.best_estimator_

print("Best parameters found for SVC: ", best_params)
print("Best model found for SVC: ", best_model)

In [ ]:
# get the predictions for the validation set
val_preds = best_model.predict(X_val['input_text_lower'])

In [ ]:
val_labels = y_val

In [ ]:
pd.crosstab(val_labels, val_preds, rownames=['Actual'], colnames=['Predicted'])

In [ ]:
from sklearn.metrics import classification_report, cohen_kappa_score
print('classification report')
print(classification_report(val_preds, val_labels))

# Test Results

In [ ]:
test_df = pd.read_csv('analyses/NOS/actors_NER/actors_with_sentences_checked.csv',
                 sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)

# change article_id to integer
test_df['article_id'] = test_df['article_id'].astype(int)
print(test_df.shape)
test_df.head()
# drop if colnames has unnamed 
test_df = test_df.loc[:, ~test_df.columns.str.contains('^Unnamed')]
test_df.columns
# keep only if directly_quoted or indirectly_quoted is 1
test_df = test_df[(test_df['directly_quoted'] == 1) | (test_df['indirectly_quoted'] == 1)]
print(test_df.shape)

In [ ]:
# change these to letters
test_df.loc[test_df.actor_function == 'NL - Nationale regering - executive / uitvoerende macht', 'actor_function'] = 1
test_df.loc[test_df.actor_function == 'NL - Nationaal parlement en nationale partijen – wetgevende macht', 'actor_function'] = 1
test_df.loc[test_df.actor_function == 'NL - Nationale regionale en lokale politieke organisaties en hun ambtenaren', 'actor_function'] = 1
test_df.loc[test_df.actor_function == 'NL - Nationale staatsorganisaties en hun ambtenaren', 'actor_function'] = 1
test_df.loc[test_df.actor_function == 'NL - Nationale koninklijke familie en haar leden', 'actor_function'] = 1
test_df.loc[test_df.actor_function == 'Regeringen/regeringsleiders/regeringsleden en/of andere politici in een ander land dan NL, op nationaal of lokaal niveau OF staatsorganisaties en hun ambtenaren', 'actor_function'] = 1
test_df.loc[test_df.actor_function == 'EU-instellingen en Internationale overheidsorganisaties (ook IGO’s) en hun leden', 'actor_function'] = 1
test_df.loc[test_df.actor_function == 'Nationale en internationale rechterlijke macht', 'actor_function'] = 1
test_df.loc[test_df.actor_function == 'Wetenschappelijke/medische organisaties en onderzoekers', 'actor_function'] = 2
test_df.loc[test_df.actor_function == 'Openbare en semiopenbare instellingen', 'actor_function'] = 2
test_df.loc[test_df.actor_function == 'Zakelijke organisaties en hun werknemers', 'actor_function'] = 2
test_df.loc[test_df.actor_function == 'Bekende mediapersonen (anders dan journalisten)', 'actor_function'] = 2
test_df.loc[test_df.actor_function == 'Journalisten anders dan de schrijver van het huidige artikel of nieuwsorganisaties anders dan de nieuwsorganisatie van het huidige artikel.', 'actor_function'] = 2
test_df.loc[test_df.actor_function == 'Niet-governementele organisaties (NGO), maatschappelijke organisaties, en hun leden', 'actor_function'] = 3
test_df.loc[test_df.actor_function == 'Religieuze instellingen en hun leden (ook gelovigen)', 'actor_function'] = 3
test_df.loc[test_df.actor_function == 'Publiek en leden van het publiek, publieke opiniepeilingen en hun respondenten', 'actor_function'] = 4

# see where actor_function is NaN
test_df[test_df['actor_function'].isna() == True]
# keep only if actor_function is not NaN
test_df = test_df[test_df['actor_function'].isna() == False]
print(test_df.shape)
# make actor_function integer
test_df['actor_function'] = test_df['actor_function'].astype(int)
test_df.actor_function.value_counts(dropna=False)

In [ ]:
test_df['input_text_lower'] = test_df['input_text_corrected'].apply(text_lower)

In [ ]:
test_preds_researcher = best_model.predict(test_df['input_text_lower'])

test_labels_researcher = test_df['actor_function']

In [ ]:
from sklearn.metrics import classification_report, cohen_kappa_score
print('classification report')
print(classification_report(test_preds_researcher, test_labels_researcher))

In [ ]:
# put the predictions in the test_df
test_df['functions_pred_SVM'] = test_preds_researcher
test_df.head()

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(test_labels_researcher, test_preds_researcher)


In [ ]:
# save the df_researcher
test_df.to_csv('functions_SVM_predictions_final.csv',
               sep = ';', encoding = 'utf-8', index = False, quoting=csv.QUOTE_NONNUMERIC)